In [ ]:
!pip -q install pandas openpyxl scikit-learn joblib

import zipfile, os, re, calendar, json
import numpy as np
import pandas as pd
from pathlib import Path

ZIP_PATH = "/content/WildLife_Data_Set.zip"   # upload your zip to Colab, then set this path
EXTRACT_DIR = "/content/wildlife_extracted"

month_names = [calendar.month_name[i] for i in range(1, 13)]
month_set = set([m.lower() for m in month_names])

def is_positive(v):
    if pd.isna(v):
        return False
    if isinstance(v, (int, float, np.number)):
        return float(v) > 0
    s = str(v).strip()
    if s in ["", "nan", "_", "-", "–", "—"]:
        return False
    try:
        return float(s) > 0
    except:
        return False

def to_float(v):
    if pd.isna(v):
        return np.nan
    if isinstance(v, (int, float, np.number)):
        return float(v)
    s = str(v).strip()
    if s in ["", "nan", "_", "-", "–", "—"]:
        return np.nan
    try:
        return float(s)
    except:
        return np.nan

def parse_workbook(path):
    xls = pd.ExcelFile(path)

    df_raw = None
    for s in xls.sheet_names:
        tmp = pd.read_excel(path, sheet_name=s, header=None)
        if tmp.shape[0] > 10 and tmp.shape[1] > 5:
            df_raw = tmp
            break
    if df_raw is None:
        return pd.DataFrame()

    hdr_row = None
    for r in range(min(15, df_raw.shape[0])):
        row_vals = df_raw.iloc[r].astype(str).str.lower().tolist()
        if any("month" in v for v in row_vals) and any("number of cases" in v for v in row_vals):
            hdr_row = r
            break
    if hdr_row is None:
        hdr_row = 0

    label_row = hdr_row + 1 if hdr_row + 1 < df_raw.shape[0] else hdr_row
    labels = df_raw.iloc[label_row].tolist()

    colnames = []
    for c in range(df_raw.shape[1]):
        if c == 0:
            colnames.append("month")
        elif c == 1:
            colnames.append("num_cases")
        elif c == 2:
            colnames.append("date_of_raid")
        else:
            v = labels[c]
            if pd.isna(v) or str(v).strip() == "":
                colnames.append(f"col_{c}")
            else:
                colnames.append(str(v).strip())

    data = df_raw.iloc[label_row + 1:].copy()
    data.columns = colnames
    data = data.dropna(how="all")

    def norm_month(x):
        if pd.isna(x):
            return np.nan
        s = str(x).strip()
        if s == "" or s.lower() == "nan":
            return np.nan
        return s

    data["month"] = data["month"].apply(norm_month)
    data = data[~data["month"].astype(str).str.lower().str.contains("total", na=False)]
    data["month"] = data["month"].ffill()
    data = data[data["month"].astype(str).str.lower().isin(month_set)]

    data["num_cases"] = pd.to_numeric(data["num_cases"], errors="coerce")
    data = data[~data["num_cases"].isna()]
    return data

def clean_offence(label):
    if label is None or (isinstance(label, float) and np.isnan(label)):
        return "Other"
    s = str(label).strip().lower()
    if s in ["unknown", "nan", ""]:
        return "Other"
    if s == "other":
        return "Other"

    s = s.replace("&", "and")

    if "illegal entrance" in s or "illegal entry" in s:
        return "Illegal Entry"
    if "wepon" in s or "explos" in s or "trap" in s or "poision" in s or "poison" in s:
        return "Weapons/Traps"
    if "timber" in s or "tree cutting" in s or "felling" in s:
        return "Illegal Logging"
    if "hunting" in s or "killing" in s:
        return "Hunting/Killing"
    if "fishing" in s:
        return "Illegal Fishing"
    if "meat" in s or "eggs" in s:
        return "Meat/Egg Trade"
    if "part of animal" in s:
        return "Animal Parts Trade"
    if "tusk" in s:
        return "Tusk Theft"
    if "domestic animal" in s:
        return "Domestic Animal Trespass"
    if "animal death" in s or "animal deth" in s or "deths" in s or "injur" in s or "accident" in s:
        if "elephant" in s:
            return "Elephant Death"
        return "Animal Death/Injury"
    if "encroch" in s or "encroach" in s or "land grabs" in s or "boundary zone" in s:
        return "Encroachment/Land"
    if "canabies" in s or "liquar" in s or "liquor" in s or "drug" in s:
        return "Drugs/Alcohol"
    if "mining" in s or "sand" in s or "treasure" in s or "mineral" in s:
        return "Mining"
    if "set fire" in s or "fire" in s:
        return "Fire"
    if "possession" in s and ("live" in s or "dead" in s or "animals" in s):
        return "Animal Possession"

    return "Other"

def derive_event_rows(df, region, year, source_file):
    cols = list(df.columns)

    start = None
    for key in ["Sand/Mineral Mining", "Treasure Mining", "Illegal Timber", "Hunting and Killing", "Illegal Fishing", "Sand"]:
        for c in cols:
            if isinstance(c, str) and key.lower() in c.lower():
                start = cols.index(c)
                break
        if start is not None:
            break
    if start is None:
        start = 3

    fauna_col = next((c for c in cols if isinstance(c, str) and c.strip().lower() == "fauna"), None)
    if fauna_col:
        end = cols.index(fauna_col) - 1
    else:
        end = len(cols) - 1

    location_cols = [c for c in cols[3:start] if not str(c).startswith("col_")]
    offence_cols = [c for c in cols[start:end + 1] if not str(c).startswith("col_") and str(c).strip().lower() != "fauna"]

    rows = []
    for _, row in df.iterrows():
        m = str(row["month"]).strip()
        num = to_float(row["num_cases"])
        if pd.isna(num):
            continue

        loc_candidates = [c for c in location_cols if is_positive(row.get(c))]
        location = loc_candidates[0] if loc_candidates else "Unknown"

        off_candidates = [c for c in offence_cols if is_positive(row.get(c))]
        offence_raw = off_candidates[0] if off_candidates else "Unknown"

        rows.append({
            "region": region,
            "year": int(year),
            "month": m,
            "month_num": month_names.index(m) + 1,
            "location": location,
            "offence_raw": offence_raw,
            "num_cases": float(num),
            "source_file": source_file
        })

    out = pd.DataFrame(rows)
    out["offence"] = out["offence_raw"].apply(clean_offence)
    out["location_id"] = out["region"].astype(str) + "|" + out["location"].astype(str)
    return out

# 1) Extract ZIP
if os.path.exists(EXTRACT_DIR):
    import shutil
    shutil.rmtree(EXTRACT_DIR)
os.makedirs(EXTRACT_DIR, exist_ok=True)

with zipfile.ZipFile(ZIP_PATH) as zf:
    xlsx_files = [n for n in zf.namelist() if n.lower().endswith(".xlsx")]
    xlsx_files = [n for n in xlsx_files if not n.startswith("__MACOSX/") and "/._" not in n]
    for n in xlsx_files:
        zf.extract(n, EXTRACT_DIR)

paths = [Path(EXTRACT_DIR) / n for n in xlsx_files]

# 2) Build event-level records
events = []
for p in paths:
    parts = p.parts
    year = next((part for part in parts if re.fullmatch(r"\d{4}", part)), None)
    region = parts[parts.index("WildLife_Data_Set") + 1] if "WildLife_Data_Set" in parts else parts[-3]

    df = parse_workbook(p)
    if df.empty:
        continue

    ev = derive_event_rows(df, region, year, str(p))
    events.append(ev)

events = pd.concat(events, ignore_index=True)
print("Event-level rows:", events.shape)

# 3) Aggregate to monthly dataset (this is what we train on)
agg = events.groupby(["location_id", "region", "location", "year", "month_num", "offence"], as_index=False)["num_cases"].sum()

total = events.groupby(["location_id", "region", "location", "year", "month_num"], as_index=False)["num_cases"].sum()
total = total.rename(columns={"num_cases": "total_cases"})

pivot = agg.pivot_table(
    index=["location_id", "region", "location", "year", "month_num"],
    columns="offence",
    values="num_cases",
    fill_value=0,
    aggfunc="sum"
).reset_index()

offence_cols = [c for c in pivot.columns if c not in ["location_id", "region", "location", "year", "month_num"]]

def top_offence(row):
    non_other = [(c, row[c]) for c in offence_cols if c != "Other" and row[c] > 0]
    if non_other:
        return max(non_other, key=lambda x: x[1])[0]
    return "Other"

pivot["top_offence"] = pivot.apply(top_offence, axis=1)

monthly = pivot.merge(total, on=["location_id", "region", "location", "year", "month_num"], how="left")
monthly["has_offence"] = (monthly["total_cases"] > 0).astype(int)

# 4) Create full grid to include months with ZERO offences (negative examples)
years = sorted(events["year"].unique())
locs = sorted(events["location_id"].unique())

grid = pd.MultiIndex.from_product([locs, years, range(1, 13)], names=["location_id", "year", "month_num"]).to_frame(index=False)
meta = events.drop_duplicates("location_id")[["location_id", "region", "location"]]
grid = grid.merge(meta, on="location_id", how="left")

full = grid.merge(monthly, on=["location_id", "region", "location", "year", "month_num"], how="left")
for c in offence_cols:
    full[c] = full[c].fillna(0.0)

full["total_cases"] = full["total_cases"].fillna(0.0)
full["has_offence"] = (full["total_cases"] > 0).astype(int)
full["top_offence"] = full["top_offence"].fillna("None")

# 5) Add time + season + history-lag features
full["date"] = pd.to_datetime(full["year"].astype(int).astype(str) + "-" + full["month_num"].astype(int).astype(str) + "-01")
full["month_sin"] = np.sin(2 * np.pi * full["month_num"] / 12)
full["month_cos"] = np.cos(2 * np.pi * full["month_num"] / 12)

def sri_lanka_season(m):
    if m in [12, 1, 2]:
        return "NE_monsoon"
    if m in [5, 6, 7, 8, 9]:
        return "SW_monsoon"
    if m in [3, 4]:
        return "Inter_MarApr"
    return "Inter_OctNov"

full["season"] = full["month_num"].apply(sri_lanka_season)

full = full.sort_values(["location_id", "year", "month_num"]).reset_index(drop=True)

for lag in [1, 2, 3]:
    full[f"lag_cases_{lag}"] = full.groupby("location_id")["total_cases"].shift(lag).fillna(0.0)
    full[f"lag_has_{lag}"] = full.groupby("location_id")["has_offence"].shift(lag).fillna(0).astype(int)
    full[f"lag_top_{lag}"] = full.groupby("location_id")["top_offence"].shift(lag).fillna("None")

full.to_csv("/content/ml_dataset_monthly.csv", index=False)
print("Saved:", "/content/ml_dataset_monthly.csv")


Event-level rows: (3481, 10)
Saved: /content/ml_dataset_monthly.csv


In [ ]:
!pip -q install scikit-learn joblib

import pandas as pd
import numpy as np
import joblib

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score, accuracy_score, classification_report

df = pd.read_csv("/content/ml_dataset_monthly.csv")

cat_features = ["region", "location", "season", "lag_top_1", "lag_top_2", "lag_top_3"]
num_features = ["month_num", "month_sin", "month_cos",
                "lag_cases_1", "lag_cases_2", "lag_cases_3",
                "lag_has_1", "lag_has_2", "lag_has_3"]

X = df[cat_features + num_features]

# Split by time: train=2022-2023, test=2024
train_idx = df["year"] < 2024
test_idx = df["year"] == 2024

X_train, X_test = X[train_idx], X[test_idx]

# 1) Risk model
y_train = df.loc[train_idx, "has_offence"]
y_test  = df.loc[test_idx, "has_offence"]

preprocess = ColumnTransformer([
    ("cat", Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("ohe", OneHotEncoder(handle_unknown="ignore"))
    ]), cat_features),
    ("num", Pipeline([
        ("imputer", SimpleImputer(strategy="median"))
    ]), num_features),
])

risk_model = Pipeline([
    ("preprocess", preprocess),
    ("model", RandomForestClassifier(
        n_estimators=400,
        random_state=42,
        class_weight="balanced",
        n_jobs=-1
    ))
])

risk_model.fit(X_train, y_train)
proba = risk_model.predict_proba(X_test)[:, 1]
pred  = risk_model.predict(X_test)

print("Risk ROC-AUC:", roc_auc_score(y_test, proba))
print("Risk Accuracy:", accuracy_score(y_test, pred))
print(classification_report(y_test, pred))

# 2) Offence type model (only where offence exists)
type_train = train_idx & (df["has_offence"] == 1)
type_test  = test_idx  & (df["has_offence"] == 1)

X2_train = df.loc[type_train, cat_features + num_features]
y2_train = df.loc[type_train, "top_offence"]

X2_test = df.loc[type_test, cat_features + num_features]
y2_test = df.loc[type_test, "top_offence"]

type_model = Pipeline([
    ("preprocess", preprocess),
    ("model", RandomForestClassifier(
        n_estimators=500,
        random_state=42,
        class_weight="balanced",
        n_jobs=-1
    ))
])

type_model.fit(X2_train, y2_train)
pred2 = type_model.predict(X2_test)

print("Type Accuracy:", accuracy_score(y2_test, pred2))

# Save models
joblib.dump(risk_model, "/content/risk_model.joblib")
joblib.dump(type_model, "/content/type_model.joblib")
print("Saved models to /content/")


Risk ROC-AUC: 0.7449681034360004
Risk Accuracy: 0.7148692810457516
              precision    recall  f1-score   support

           0       0.82      0.80      0.81       918
           1       0.43      0.46      0.45       306

    accuracy                           0.71      1224
   macro avg       0.62      0.63      0.63      1224
weighted avg       0.72      0.71      0.72      1224

Type Accuracy: 0.2549019607843137
Saved models to /content/


In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv("/content/ml_dataset_monthly.csv")

df = df.sort_values(["location_id", "year", "month_num"]).reset_index(drop=True)

# Rolling averages
df["roll_3_cases"] = (
    df.groupby("location_id")["total_cases"]
      .rolling(3, min_periods=1)
      .mean()
      .reset_index(level=0, drop=True)
)

df["roll_6_cases"] = (
    df.groupby("location_id")["total_cases"]
      .rolling(6, min_periods=1)
      .mean()
      .reset_index(level=0, drop=True)
)

# Trends (difference between now and N months ago)
df["trend_3"] = df["total_cases"] - df.groupby("location_id")["total_cases"].shift(3)
df["trend_6"] = df["total_cases"] - df.groupby("location_id")["total_cases"].shift(6)

df[["trend_3", "trend_6"]] = df[["trend_3", "trend_6"]].fillna(0)

print("Added time-based features")


Added time-based features


In [ ]:
expanded = []

for shift in [1, 2]:
    temp = df.copy()
    temp["year"] = temp["year"] - (shift // 12)
    temp["month_num"] = temp["month_num"] - shift
    temp = temp[temp["month_num"] > 0]
    temp["synthetic_flag"] = 1
    expanded.append(temp)

df["synthetic_flag"] = 0
df_expanded = pd.concat([df] + expanded, ignore_index=True)

print("Rows before:", len(df))
print("Rows after lag expansion:", len(df_expanded))


Rows before: 3672
Rows after lag expansion: 10098


In [ ]:
np.random.seed(42)

spatial_rows = []

for _, row in df.iterrows():
    same_region = df[
        (df["region"] == row["region"]) &
        (df["location"] != row["location"])
    ]

    if len(same_region) == 0:
        continue

    sample = same_region.sample(1).iloc[0].copy()

    for col in ["total_cases", "roll_3_cases", "roll_6_cases"]:
        sample[col] = max(0, sample[col] + np.random.normal(0, 0.3))

    sample["synthetic_flag"] = 2
    spatial_rows.append(sample)

df_spatial = pd.concat([df_expanded, pd.DataFrame(spatial_rows)], ignore_index=True)

print("Rows after spatial expansion:", len(df_spatial))


Rows after spatial expansion: 13770


In [ ]:
from sklearn.utils import resample

rare_classes = df_spatial["top_offence"].value_counts()
rare_classes = rare_classes[rare_classes < 50].index.tolist()

balanced_rows = []

for cls in rare_classes:
    cls_df = df_spatial[df_spatial["top_offence"] == cls]
    if len(cls_df) < 10:
        continue

    boosted = resample(
        cls_df,
        replace=True,
        n_samples=50,
        random_state=42
    )
    boosted["synthetic_flag"] = 3
    balanced_rows.append(boosted)

df_final = pd.concat([df_spatial] + balanced_rows, ignore_index=True)

print("Final expanded dataset size:", len(df_final))


Final expanded dataset size: 13970


In [ ]:
df_final.to_csv("/content/ml_dataset_expanded.csv", index=False)
print("Saved expanded dataset")


Saved expanded dataset


In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv("/content/ml_dataset_expanded.csv")

print("Total rows:", len(df))
print(df["synthetic_flag"].value_counts())


Total rows: 13970
synthetic_flag
1    6426
0    3672
2    3672
3     200
Name: count, dtype: int64


In [ ]:
cat_features = [
    "region",
    "location",
    "season",
    "lag_top_1",
    "lag_top_2",
    "lag_top_3"
]

num_features = [
    "month_num",
    "month_sin",
    "month_cos",
    "lag_cases_1",
    "lag_cases_2",
    "lag_cases_3",
    "lag_has_1",
    "lag_has_2",
    "lag_has_3",
    "roll_3_cases",
    "roll_6_cases",
    "trend_3",
    "trend_6"
]

X = df[cat_features + num_features]

y_risk = df["has_offence"]
y_type = df["top_offence"]


In [ ]:
train_mask = df["year"] < 2024
test_mask  = (df["year"] == 2024) & (df["synthetic_flag"] == 0)

X_train = X[train_mask]
X_test  = X[test_mask]

y_risk_train = y_risk[train_mask]
y_risk_test  = y_risk[test_mask]

print("Train rows:", len(X_train))
print("Test rows (real only):", len(X_test))


Train rows: 9329
Test rows (real only): 1224


In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer


In [ ]:
preprocess = ColumnTransformer(
    transformers=[
        ("cat", Pipeline([
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("onehot", OneHotEncoder(handle_unknown="ignore"))
        ]), cat_features),

        ("num", Pipeline([
            ("imputer", SimpleImputer(strategy="median"))
        ]), num_features)
    ]
)


In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score, classification_report

risk_model = Pipeline([
    ("preprocess", preprocess),
    ("model", RandomForestClassifier(
        n_estimators=600,
        max_depth=18,
        min_samples_leaf=3,
        class_weight="balanced",
        random_state=42,
        n_jobs=-1
    ))
])

risk_model.fit(X_train, y_risk_train)

risk_probs = risk_model.predict_proba(X_test)[:, 1]
risk_preds = (risk_probs > 0.5).astype(int)

print("RISK MODEL ROC-AUC:", roc_auc_score(y_risk_test, risk_probs))
print(classification_report(y_risk_test, risk_preds))


RISK MODEL ROC-AUC: 0.9974938414000314
              precision    recall  f1-score   support

           0       0.99      0.97      0.98       918
           1       0.92      0.97      0.95       306

    accuracy                           0.97      1224
   macro avg       0.96      0.97      0.96      1224
weighted avg       0.97      0.97      0.97      1224



In [ ]:
type_train_mask = train_mask & (df["has_offence"] == 1)
type_test_mask  = test_mask  & (df["has_offence"] == 1)

X_type_train = X[type_train_mask]
X_type_test  = X[type_test_mask]

y_type_train = y_type[type_train_mask]
y_type_test  = y_type[type_test_mask]


In [ ]:
type_model = Pipeline([
    ("preprocess", preprocess),
    ("model", RandomForestClassifier(
        n_estimators=700,
        max_depth=20,
        min_samples_leaf=2,
        class_weight="balanced",
        random_state=42,
        n_jobs=-1
    ))
])

type_model.fit(X_type_train, y_type_train)

type_preds = type_model.predict(X_type_test)

from sklearn.metrics import accuracy_score

print("TYPE MODEL ACCURACY:", accuracy_score(y_type_test, type_preds))
print(classification_report(y_type_test, type_preds))


TYPE MODEL ACCURACY: 0.24183006535947713
                          precision    recall  f1-score   support

     Animal Death/Injury       0.00      0.00      0.00         2
      Animal Parts Trade       0.18      0.45      0.26        11
       Animal Possession       0.00      0.00      0.00        16
Domestic Animal Trespass       0.40      0.22      0.29         9
           Drugs/Alcohol       0.00      0.00      0.00         1
       Encroachment/Land       0.00      0.00      0.00        17
                    Fire       0.00      0.00      0.00         2
         Hunting/Killing       0.08      0.12      0.10         8
           Illegal Entry       0.25      0.23      0.24        30
         Illegal Fishing       0.00      0.00      0.00        14
         Illegal Logging       0.07      0.12      0.09         8
          Meat/Egg Trade       0.37      0.37      0.37        79
                  Mining       0.00      0.00      0.00         0
                   Other       0.4

/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_

In [ ]:
import joblib

joblib.dump(risk_model, "/content/risk_model_v2.joblib")
joblib.dump(type_model, "/content/type_model_v2.joblib")

print("Saved:")
print(" - risk_model_v2.joblib")
print(" - type_model_v2.joblib")


Saved:
 - risk_model_v2.joblib
 - type_model_v2.joblib


In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv("/content/ml_dataset_monthly.csv")

# Always sort before creating time-based features
df = df.sort_values(["location_id", "year", "month_num"]).reset_index(drop=True)

# Rolling averages (includes current row month by default)
df["roll_3_cases"] = (
    df.groupby("location_id")["total_cases"]
      .rolling(3, min_periods=1)
      .mean()
      .reset_index(level=0, drop=True)
)

df["roll_6_cases"] = (
    df.groupby("location_id")["total_cases"]
      .rolling(6, min_periods=1)
      .mean()
      .reset_index(level=0, drop=True)
)

# Trends (current month compared to N months ago)
df["trend_3"] = df["total_cases"] - df.groupby("location_id")["total_cases"].shift(3)
df["trend_6"] = df["total_cases"] - df.groupby("location_id")["total_cases"].shift(6)

df["trend_3"] = df["trend_3"].fillna(0.0)
df["trend_6"] = df["trend_6"].fillna(0.0)

print(df[["location_id","year","month_num","total_cases","roll_3_cases","roll_6_cases","trend_3","trend_6"]].head(10))


                    location_id  year  month_num  total_cases  roll_3_cases  \
0  Anuradhapura|Horowpothana NP  2022          1          2.0      2.000000   
1  Anuradhapura|Horowpothana NP  2022          2          0.0      1.000000   
2  Anuradhapura|Horowpothana NP  2022          3         12.0      4.666667   
3  Anuradhapura|Horowpothana NP  2022          4         21.0     11.000000   
4  Anuradhapura|Horowpothana NP  2022          5          0.0     11.000000   
5  Anuradhapura|Horowpothana NP  2022          6          0.0      7.000000   
6  Anuradhapura|Horowpothana NP  2022          7         23.0      7.666667   
7  Anuradhapura|Horowpothana NP  2022          8         27.0     16.666667   
8  Anuradhapura|Horowpothana NP  2022          9          0.0     16.666667   
9  Anuradhapura|Horowpothana NP  2022         10          0.0      9.000000   

   roll_6_cases  trend_3  trend_6  
0      2.000000      0.0      0.0  
1      1.000000      0.0      0.0  
2      4.666667      0

In [ ]:
!pip -q install scikit-learn joblib

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score, accuracy_score, classification_report
import joblib

# Labels
y_risk = df["has_offence"]
y_type = df["top_offence"]

# Feature columns
cat_features = ["region", "location", "season", "lag_top_1", "lag_top_2", "lag_top_3"]

num_features = [
    "month_num", "month_sin", "month_cos",
    "lag_cases_1", "lag_cases_2", "lag_cases_3",
    "lag_has_1", "lag_has_2", "lag_has_3",
    "roll_3_cases", "roll_6_cases",
    "trend_3", "trend_6"
]

X = df[cat_features + num_features]

# Time split example (edit if your years differ)
train_mask = df["year"] < 2024
test_mask  = df["year"] == 2024

X_train, X_test = X[train_mask], X[test_mask]

# Preprocess
preprocess = ColumnTransformer([
    ("cat", Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("ohe", OneHotEncoder(handle_unknown="ignore"))
    ]), cat_features),

    ("num", Pipeline([
        ("imputer", SimpleImputer(strategy="median"))
    ]), num_features)
])

# 1) Risk model
risk_model = Pipeline([
    ("preprocess", preprocess),
    ("model", RandomForestClassifier(
        n_estimators=600,
        max_depth=18,
        min_samples_leaf=3,
        class_weight="balanced",
        random_state=42,
        n_jobs=-1
    ))
])

risk_model.fit(X_train, y_risk[train_mask])

risk_probs = risk_model.predict_proba(X_test)[:, 1]
risk_preds = (risk_probs >= 0.5).astype(int)

print("Risk ROC-AUC:", roc_auc_score(y_risk[test_mask], risk_probs))
print(classification_report(y_risk[test_mask], risk_preds))

# 2) Type model (only rows with offences)
type_train = train_mask & (df["has_offence"] == 1)
type_test  = test_mask  & (df["has_offence"] == 1)

type_model = Pipeline([
    ("preprocess", preprocess),
    ("model", RandomForestClassifier(
        n_estimators=700,
        max_depth=20,
        min_samples_leaf=2,
        class_weight="balanced",
        random_state=42,
        n_jobs=-1
    ))
])

type_model.fit(X[type_train], y_type[type_train])

type_preds = type_model.predict(X[type_test])

print("Type Accuracy:", accuracy_score(y_type[type_test], type_preds))
print(classification_report(y_type[type_test], type_preds))

# Save
joblib.dump(risk_model, "/content/risk_model_v2.joblib")
joblib.dump(type_model, "/content/type_model_v2.joblib")

# Save the dataset too (so FastAPI can look up history)
df.to_csv("/content/ml_dataset_with_roll_trend.csv", index=False)

print("Saved: risk_model_v2.joblib, type_model_v2.joblib, ml_dataset_with_roll_trend.csv")


Risk ROC-AUC: 0.9969562988594132
              precision    recall  f1-score   support

           0       1.00      0.94      0.97       918
           1       0.84      0.99      0.91       306

    accuracy                           0.95      1224
   macro avg       0.92      0.96      0.94      1224
weighted avg       0.96      0.95      0.95      1224

Type Accuracy: 0.21241830065359477
                          precision    recall  f1-score   support

     Animal Death/Injury       0.00      0.00      0.00         2
      Animal Parts Trade       0.14      0.55      0.23        11
       Animal Possession       0.00      0.00      0.00        16
Domestic Animal Trespass       0.18      0.22      0.20         9
           Drugs/Alcohol       0.25      1.00      0.40         1
       Encroachment/Land       0.00      0.00      0.00        17
                    Fire       0.00      0.00      0.00         2
         Hunting/Killing       0.00      0.00      0.00         8
          

/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_

Saved: risk_model_v2.joblib, type_model_v2.joblib, ml_dataset_with_roll_trend.csv


In [ ]:
# ----------------------------
# 0) Install + imports
# ----------------------------
!pip -q install pandas numpy scikit-learn joblib

import json
import numpy as np
import pandas as pd
import joblib

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score, accuracy_score, classification_report, confusion_matrix


# ----------------------------
# 1) Load dataset
# ----------------------------
DATA_PATH = "/content/ml_dataset_expanded.csv"
df = pd.read_csv(DATA_PATH)

# Basic cleanup / type safety
for c in ["year", "month_num"]:
    df[c] = pd.to_numeric(df[c], errors="coerce").astype("Int64")

df["total_cases"] = pd.to_numeric(df["total_cases"], errors="coerce").fillna(0.0).astype(float)
df["has_offence"] = pd.to_numeric(df["has_offence"], errors="coerce").fillna(0).astype(int)

# Some expanded files may have missing synthetic_flag (just in case)
if "synthetic_flag" not in df.columns:
    df["synthetic_flag"] = 0
df["synthetic_flag"] = pd.to_numeric(df["synthetic_flag"], errors="coerce").fillna(0).astype(int)

# Ensure sort for time features
df = df.sort_values(["location_id", "year", "month_num"]).reset_index(drop=True)

print("Rows:", len(df))
print("synthetic_flag counts:\n", df["synthetic_flag"].value_counts())


# ----------------------------
# 2) Forecast-safe rolling + trend features
# ----------------------------
# Forecast-safe rule:
# - For predicting month T, rolling/trend must use months < T only.
# So we shift total_cases by 1 month within each location.
prev_cases = df.groupby("location_id")["total_cases"].shift(1).fillna(0.0)

# Rolling averages using ONLY previous months
df["roll_3_cases"] = (
    prev_cases.groupby(df["location_id"])
    .rolling(3, min_periods=1)
    .mean()
    .reset_index(level=0, drop=True)
)

df["roll_6_cases"] = (
    prev_cases.groupby(df["location_id"])
    .rolling(6, min_periods=1)
    .mean()
    .reset_index(level=0, drop=True)
)

# Trends using ONLY previous months:
# trend_3 = cases(last_month) - cases(3 months before last_month)
# which is: prev_cases - prev_cases.shift(3)
df["trend_3"] = prev_cases - prev_cases.groupby(df["location_id"]).shift(3).fillna(0.0)
df["trend_6"] = prev_cases - prev_cases.groupby(df["location_id"]).shift(6).fillna(0.0)

# Any remaining NaNs -> 0
for c in ["roll_3_cases", "roll_6_cases", "trend_3", "trend_6"]:
    df[c] = pd.to_numeric(df[c], errors="coerce").fillna(0.0).astype(float)

print("\nForecast-safe features added:",
      ["roll_3_cases","roll_6_cases","trend_3","trend_6"])


# ----------------------------
# 3) Feature columns (match FastAPI)
# ----------------------------
cat_features = [
    "region",
    "location",
    "season",
    "lag_top_1",
    "lag_top_2",
    "lag_top_3"
]

num_features = [
    "month_num", "month_sin", "month_cos",
    "lag_cases_1", "lag_cases_2", "lag_cases_3",
    "lag_has_1", "lag_has_2", "lag_has_3",
    "roll_3_cases", "roll_6_cases",
    "trend_3", "trend_6"
]

# Verify columns exist
missing = [c for c in (cat_features + num_features + ["has_offence","top_offence","year"]) if c not in df.columns]
if missing:
    raise ValueError(f"Missing required columns in dataset: {missing}")

X = df[cat_features + num_features]
y_risk = df["has_offence"]
y_type = df["top_offence"].astype(str)


# ----------------------------
# 4) Train/Test split (time-aware + real-only evaluation)
# ----------------------------
# Train on everything BEFORE the test year.
# Evaluate on test year but ONLY real rows (synthetic_flag == 0).
TEST_YEAR = int(df["year"].dropna().max())  # uses latest year in your data

train_mask = df["year"] < TEST_YEAR
test_mask  = (df["year"] == TEST_YEAR) & (df["synthetic_flag"] == 0)

X_train, X_test = X[train_mask], X[test_mask]
y_risk_train, y_risk_test = y_risk[train_mask], y_risk[test_mask]

print("\nTrain rows:", len(X_train))
print("Test rows (real only):", len(X_test))
print("Test year:", TEST_YEAR)

if len(X_test) < 30:
    print("\nWarning: very small real-only test set. Consider using last 2 years as test.")


# ----------------------------
# 5) Preprocessing pipeline
# ----------------------------
preprocess = ColumnTransformer(
    transformers=[
        ("cat", Pipeline([
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("ohe", OneHotEncoder(handle_unknown="ignore"))
        ]), cat_features),
        ("num", Pipeline([
            ("imputer", SimpleImputer(strategy="median"))
        ]), num_features),
    ],
    remainder="drop"
)


# ----------------------------
# 6) Train Risk model (probability of offence)
# ----------------------------
risk_model = Pipeline([
    ("preprocess", preprocess),
    ("model", RandomForestClassifier(
        n_estimators=700,
        max_depth=20,
        min_samples_leaf=2,
        class_weight="balanced",
        random_state=42,
        n_jobs=-1
    ))
])

risk_model.fit(X_train, y_risk_train)

risk_probs = risk_model.predict_proba(X_test)[:, 1]
risk_pred  = (risk_probs >= 0.5).astype(int)

print("\n=== RISK MODEL (Real-only test) ===")
print("ROC-AUC:", roc_auc_score(y_risk_test, risk_probs))
print("Confusion matrix:\n", confusion_matrix(y_risk_test, risk_pred))
print(classification_report(y_risk_test, risk_pred, digits=4))


# ----------------------------
# 7) Train Type model (predict top offence type when offence occurs)
# ----------------------------
type_train_mask = train_mask & (df["has_offence"] == 1)
type_test_mask  = test_mask  & (df["has_offence"] == 1)

X_type_train = X[type_train_mask]
y_type_train = y_type[type_train_mask]

X_type_test  = X[type_test_mask]
y_type_test  = y_type[type_test_mask]

type_model = Pipeline([
    ("preprocess", preprocess),
    ("model", RandomForestClassifier(
        n_estimators=900,
        max_depth=24,
        min_samples_leaf=2,
        class_weight="balanced",
        random_state=42,
        n_jobs=-1
    ))
])

type_model.fit(X_type_train, y_type_train)

type_pred = type_model.predict(X_type_test)

print("\n=== TYPE MODEL (Real-only test, offence months only) ===")
print("Accuracy:", accuracy_score(y_type_test, type_pred))
print(classification_report(y_type_test, type_pred, digits=4))


# ----------------------------
# 8) Save models + metadata for FastAPI
# ----------------------------
OUT_RISK = "/content/risk_model_v2.joblib"
OUT_TYPE = "/content/type_model_v2.joblib"
OUT_DATA = "/content/ml_dataset_expanded_forecastsafe.csv"
OUT_META = "/content/model_meta_v2.json"

joblib.dump(risk_model, OUT_RISK)
joblib.dump(type_model, OUT_TYPE)

# Save the dataset with forecast-safe columns included (FastAPI can use it too)
df.to_csv(OUT_DATA, index=False)

meta = {
    "test_year": TEST_YEAR,
    "categorical_features": cat_features,
    "numeric_features": num_features,
    "labels": {
        "risk": "has_offence",
        "type": "top_offence"
    },
    "notes": [
        "roll_3_cases, roll_6_cases, trend_3, trend_6 computed forecast-safe using only past months (shifted by 1).",
        "Evaluation performed only on real rows where synthetic_flag == 0."
    ]
}

with open(OUT_META, "w") as f:
    json.dump(meta, f, indent=2)

print("\nSaved:")
print(" -", OUT_RISK)
print(" -", OUT_TYPE)
print(" -", OUT_DATA)
print(" -", OUT_META)


Rows: 13970
synthetic_flag counts:
 synthetic_flag
1    6426
0    3672
2    3672
3     200
Name: count, dtype: int64

Forecast-safe features added: ['roll_3_cases', 'roll_6_cases', 'trend_3', 'trend_6']

Train rows: 9329
Test rows (real only): 1224
Test year: 2024

=== RISK MODEL (Real-only test) ===
ROC-AUC: 0.9190553490822617
Confusion matrix:
 [[776 142]
 [ 35 271]]
              precision    recall  f1-score   support

           0     0.9568    0.8453    0.8976       918
           1     0.6562    0.8856    0.7538       306

    accuracy                         0.8554      1224
   macro avg     0.8065    0.8655    0.8257      1224
weighted avg     0.8817    0.8554    0.8617      1224


=== TYPE MODEL (Real-only test, offence months only) ===
Accuracy: 0.23529411764705882
                          precision    recall  f1-score   support

     Animal Death/Injury     0.0500    0.5000    0.0909         2
      Animal Parts Trade     0.1765    0.2727    0.2143        11
       Animal 

/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))



Saved:
 - /content/risk_model_v2.joblib
 - /content/type_model_v2.joblib
 - /content/ml_dataset_expanded_forecastsafe.csv
 - /content/model_meta_v2.json


In [ ]:
# ============================================================
# V3 TRAINING SCRIPT (CLEAN + CONSISTENT ARTIFACTS)
# - Trains TWO pipelines end-to-end (preprocess + model together)
# - Saves matching artifacts:
#     artifacts/model/risk_model_v3.joblib
#     artifacts/model/type_model_v3.joblib
#     artifacts/model/model_meta_v3.json
# - Uses forecast-safe features (roll/trend) that are already in
#   ml_dataset_expanded_forecastsafe.csv
# ============================================================

import os
import json
import joblib
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    roc_auc_score, average_precision_score, classification_report,
    confusion_matrix
)

# -----------------------------
# 0) CONFIG
# -----------------------------
DATA_PATH = "ml_dataset_expanded_forecastsafe.csv"   # put this file in your Colab/workdir
OUT_DIR = "artifacts/model"
os.makedirs(OUT_DIR, exist_ok=True)

RISK_MODEL_OUT = os.path.join(OUT_DIR, "risk_model_v3.joblib")
TYPE_MODEL_OUT = os.path.join(OUT_DIR, "type_model_v3.joblib")
META_OUT = os.path.join(OUT_DIR, "model_meta_v3.json")

RANDOM_STATE = 42

# -----------------------------
# 1) LOAD DATA
# -----------------------------
df = pd.read_csv(DATA_PATH)

# -----------------------------
# 2) DEFINE RAW FEATURES (MUST MATCH FASTAPI)
# -----------------------------
CATEGORICAL = [
    "region",
    "location",
    "season",
    "lag_top_1",
    "lag_top_2",
    "lag_top_3",
]

NUMERIC = [
    "month_num", "month_sin", "month_cos",
    "lag_cases_1", "lag_cases_2", "lag_cases_3",
    "lag_has_1", "lag_has_2", "lag_has_3",
    "roll_3_cases", "roll_6_cases",
    "trend_3", "trend_6",
]

RISK_LABEL = "has_offence"
TYPE_LABEL = "top_offence"

REQUIRED_COLS = CATEGORICAL + NUMERIC + [RISK_LABEL, TYPE_LABEL]

missing = [c for c in REQUIRED_COLS if c not in df.columns]
if missing:
    raise ValueError(f"Dataset missing required columns: {missing}")

# -----------------------------
# 3) BASIC CLEANING / TYPE FIXES
# -----------------------------
# Categorical to string
for c in CATEGORICAL:
    df[c] = df[c].fillna("None").astype(str)

# Numeric to float (or int-like floats)
for c in NUMERIC:
    df[c] = pd.to_numeric(df[c], errors="coerce").fillna(0.0).astype(float)

# Risk label to int {0,1}
df[RISK_LABEL] = pd.to_numeric(df[RISK_LABEL], errors="coerce").fillna(0).astype(int)
df[RISK_LABEL] = df[RISK_LABEL].clip(0, 1)

# Type label to string
df[TYPE_LABEL] = df[TYPE_LABEL].fillna("None").astype(str)

# Optional: remove impossible months if present
if "month_num" in df.columns:
    df = df[(df["month_num"] >= 1) & (df["month_num"] <= 12)].copy()

# -----------------------------
# 4) BUILD X / y
# -----------------------------
X = df[CATEGORICAL + NUMERIC].copy()
y_risk = df[RISK_LABEL].copy()
y_type = df[TYPE_LABEL].copy()

# -----------------------------
# 5) SPLIT DATA (SAME SPLIT FOR BOTH TASKS)
# -----------------------------
X_train, X_test, y_risk_train, y_risk_test, y_type_train, y_type_test = train_test_split(
    X, y_risk, y_type,
    test_size=0.2,
    random_state=RANDOM_STATE,
    stratify=y_risk  # keeps offence/non-offence ratio stable
)

# -----------------------------
# 6) PIPELINE FACTORY (IMPORTANT)
# Build a NEW preprocessor for EACH pipeline.
# Do not reuse the same ColumnTransformer instance.
# -----------------------------
def make_preprocessor():
    return ColumnTransformer(
        transformers=[
            ("cat", OneHotEncoder(handle_unknown="ignore"), CATEGORICAL),
            ("num", "passthrough", NUMERIC),
        ],
        remainder="drop"
    )

def make_risk_pipeline():
    return Pipeline(steps=[
        ("preprocess", make_preprocessor()),
        ("model", RandomForestClassifier(
            n_estimators=400,
            max_depth=14,
            min_samples_split=4,
            min_samples_leaf=2,
            class_weight="balanced",
            random_state=RANDOM_STATE,
            n_jobs=-1
        )),
    ])

def make_type_pipeline():
    return Pipeline(steps=[
        ("preprocess", make_preprocessor()),
        ("model", RandomForestClassifier(
            n_estimators=500,
            max_depth=18,
            min_samples_split=3,
            min_samples_leaf=1,
            class_weight=None,
            random_state=RANDOM_STATE,
            n_jobs=-1
        )),
    ])

risk_model = make_risk_pipeline()
type_model = make_type_pipeline()

# -----------------------------
# 7) TRAIN
# -----------------------------
print("Training risk model (has_offence)...")
risk_model.fit(X_train, y_risk_train)

print("Training type model (top_offence)...")
type_model.fit(X_train, y_type_train)

# -----------------------------
# 8) EVALUATE
# -----------------------------
print("\n=== RISK MODEL EVALUATION ===")
risk_proba = risk_model.predict_proba(X_test)[:, 1]
risk_pred = (risk_proba >= 0.5).astype(int)

try:
    auc = roc_auc_score(y_risk_test, risk_proba)
    ap = average_precision_score(y_risk_test, risk_proba)
    print(f"ROC-AUC: {auc:.4f}")
    print(f"Avg Precision (PR-AUC): {ap:.4f}")
except Exception as e:
    print("Risk metrics error:", e)

print("Confusion Matrix:\n", confusion_matrix(y_risk_test, risk_pred))
print("Classification Report:\n", classification_report(y_risk_test, risk_pred, digits=4))

print("\n=== TYPE MODEL EVALUATION ===")
type_pred = type_model.predict(X_test)
print("Classification Report:\n", classification_report(y_type_test, type_pred, digits=4))

# -----------------------------
# 9) SAVE ARTIFACTS (CONSISTENT BUNDLE)
# -----------------------------
joblib.dump(risk_model, RISK_MODEL_OUT)
joblib.dump(type_model, TYPE_MODEL_OUT)

meta = {
    "version": "v3",
    "categorical_features": CATEGORICAL,
    "numeric_features": NUMERIC,
    "labels": {
        "risk": RISK_LABEL,
        "type": TYPE_LABEL
    },
    "notes": {
        "forecast_safe": True,
        "rolling_features": ["roll_3_cases", "roll_6_cases"],
        "trend_features": ["trend_3", "trend_6"]
    }
}

with open(META_OUT, "w") as f:
    json.dump(meta, f, indent=2)

print("\nSaved:")
print(" -", RISK_MODEL_OUT)
print(" -", TYPE_MODEL_OUT)
print(" -", META_OUT)

# -----------------------------
# 10) QUICK SANITY CHECK (FEATURE DIMENSIONS MATCH)
# -----------------------------
# This verifies preprocess output features == classifier expected features
def check_pipeline(p: Pipeline, name: str):
    pre = p.named_steps["preprocess"]
    clf = p.named_steps["model"]
    Xt = pre.transform(X_test.head(5))
    n_out = Xt.shape[1]
    n_in = getattr(clf, "n_features_in_", None)
    print(f"\n{name} sanity:")
    print(" preprocessor output features:", n_out)
    print(" classifier expects features:", n_in)
    if n_in is not None and n_out != n_in:
        raise RuntimeError(f"{name} mismatch: preprocess={n_out} vs model expects={n_in}")

check_pipeline(risk_model, "RISK")
check_pipeline(type_model, "TYPE")

print("\nAll good. These v3 artifacts will not produce the 134 vs 141 error.")


Training risk model (has_offence)...
Training type model (top_offence)...

=== RISK MODEL EVALUATION ===
ROC-AUC: 0.9524
Avg Precision (PR-AUC): 0.9115
Confusion Matrix:
 [[1705  345]
 [  73  671]]
Classification Report:
               precision    recall  f1-score   support

           0     0.9589    0.8317    0.8908      2050
           1     0.6604    0.9019    0.7625       744

    accuracy                         0.8504      2794
   macro avg     0.8097    0.8668    0.8267      2794
weighted avg     0.8795    0.8504    0.8566      2794


=== TYPE MODEL EVALUATION ===
Classification Report:
                           precision    recall  f1-score   support

     Animal Death/Injury     1.0000    0.5938    0.7451        32
      Animal Parts Trade     1.0000    0.4643    0.6341        28
       Animal Possession     1.0000    0.6190    0.7647        21
Domestic Animal Trespass     1.0000    0.3333    0.5000        18
           Drugs/Alcohol     1.0000    1.0000    1.0000        17

In [ ]:
from google.colab import files
uploaded = files.upload()
print("Uploaded files:", list(uploaded.keys()))



Saving ml_dataset_expanded_forecastsafe.csv to ml_dataset_expanded_forecastsafe.csv
Uploaded files: ['ml_dataset_expanded_forecastsafe.csv']


In [ ]:
import pandas as pd

DATA_PATH = "ml_dataset_expanded_forecastsafe.csv"
df = pd.read_csv(DATA_PATH)
print("Shape:", df.shape)
print("Columns:", df.columns.tolist()[:30], "...")
df.head()


Shape: (13970, 42)
Columns: ['location_id', 'year', 'month_num', 'region', 'location', 'Animal Death/Injury', 'Animal Parts Trade', 'Animal Possession', 'Domestic Animal Trespass', 'Drugs/Alcohol', 'Encroachment/Land', 'Fire', 'Hunting/Killing', 'Illegal Entry', 'Illegal Fishing', 'Illegal Logging', 'Meat/Egg Trade', 'Mining', 'Other', 'Tusk Theft', 'Weapons/Traps', 'top_offence', 'total_cases', 'has_offence', 'date', 'month_sin', 'month_cos', 'season', 'lag_cases_1', 'lag_has_1'] ...


,location_id,year,month_num,region,location,Animal Death/Injury,Animal Parts Trade,Animal Possession,Domestic Animal Trespass,Drugs/Alcohol,...,lag_has_2,lag_top_2,lag_cases_3,lag_has_3,lag_top_3,roll_3_cases,roll_6_cases,trend_3,trend_6,synthetic_flag
0,Anuradhapura|Horowpothana NP,2022,1,Anuradhapura,Horowpothana NP,0.0,0.0,0.0,0.0,0.0,...,0,NaN,0.0,0,NaN,0.000000,0.000000,0.0,0.0,0
1,Anuradhapura|Horowpothana NP,2022,1,Anuradhapura,Horowpothana NP,0.0,0.0,0.0,0.0,0.0,...,0,NaN,0.0,0,NaN,1.000000,1.000000,2.0,2.0,1
2,Anuradhapura|Horowpothana NP,2022,1,Anuradhapura,Horowpothana NP,0.0,0.0,0.0,0.0,0.0,...,1,Meat/Egg Trade,0.0,0,NaN,0.666667,0.666667,0.0,0.0,1
3,Anuradhapura|Horowpothana NP,2022,2,Anuradhapura,Horowpothana NP,0.0,0.0,0.0,0.0,0.0,...,0,NaN,0.0,0,NaN,4.666667,3.500000,12.0,12.0,0
4,Anuradhapura|Horowpothana NP,2022,2,Anuradhapura,Horowpothana NP,0.0,0.0,0.0,0.0,0.0,...,1,Meat/Egg Trade,0.0,0,NaN,4.000000,2.800000,-2.0,0.0,1


In [ ]:
import json
import joblib
import numpy as np
from pathlib import Path

from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import classification_report

# ===============================
# CONFIG
# ===============================
OUT_DIR = Path("artifacts_v4")
OUT_DIR.mkdir(exist_ok=True)

RISK_MODEL_PATH = OUT_DIR / "risk_model_v4.joblib"
TYPE_MODEL_PATH = OUT_DIR / "type_model_v4.joblib"
META_PATH = OUT_DIR / "model_meta_v4.json"

RANDOM_STATE = 42
MIN_SAMPLES_PER_CLASS = 25
RISK_THRESHOLD = 0.10

# ===============================
# FEATURES (same structure as your v3 meta)
# ===============================
CATEGORICAL_FEATURES = [
    "region", "location", "season",
    "lag_top_1", "lag_top_2", "lag_top_3"
]

NUMERIC_FEATURES = [
    "month_num", "month_sin", "month_cos",
    "lag_cases_1", "lag_has_1",
    "lag_cases_2", "lag_has_2",
    "lag_cases_3", "lag_has_3",
    "roll_3_cases", "roll_6_cases",
    "trend_3", "trend_6"
]

FEATURE_COLUMNS = CATEGORICAL_FEATURES + NUMERIC_FEATURES

# ===============================
# VALIDATE REQUIRED COLUMNS
# ===============================
required = ["year", "n_cases", "top_offence"] + FEATURE_COLUMNS
missing = [c for c in required if c not in df.columns]
if missing:
    raise ValueError(f"Missing required columns: {missing}")

# ===============================
# CREATE RISK LABEL
# ===============================
df["has_offence"] = (df["n_cases"] > 0).astype(int)

# ===============================
# TIME-BASED SPLIT (last year test)
# ===============================
max_year = int(df["year"].max())
train_df = df[df["year"] < max_year].copy()
test_df  = df[df["year"] == max_year].copy()

print("Max year:", max_year)
print("Risk train:", train_df.shape, "Risk test:", test_df.shape)

# ===============================
# PREPROCESSOR
# ===============================
preprocessor = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(handle_unknown="ignore"), CATEGORICAL_FEATURES),
        ("num", "passthrough", NUMERIC_FEATURES),
    ],
    remainder="drop",
)

# ===============================
# TRAIN RISK MODEL (binary) + calibration
# ===============================
X_train_r = train_df[FEATURE_COLUMNS]
y_train_r = train_df["has_offence"].astype(int)

X_test_r = test_df[FEATURE_COLUMNS]
y_test_r = test_df["has_offence"].astype(int)

base_risk = RandomForestClassifier(
    n_estimators=600,
    max_depth=18,
    min_samples_leaf=2,
    class_weight="balanced",
    random_state=RANDOM_STATE,
    n_jobs=-1,
)

risk_clf = CalibratedClassifierCV(
    base_estimator=base_risk,
    method="sigmoid",
    cv=3
)

risk_model = Pipeline([
    ("prep", preprocessor),
    ("clf", risk_clf),
])

print("\nTraining risk_model_v4...")
risk_model.fit(X_train_r, y_train_r)

risk_pred = risk_model.predict(X_test_r)
print("\nRISK MODEL REPORT")
print(classification_report(y_test_r, risk_pred, zero_division=0))

joblib.dump(risk_model, RISK_MODEL_PATH)
print("Saved:", RISK_MODEL_PATH)

# ===============================
# TRAIN TYPE MODEL (offence months only)
# ===============================
df_type = df[df["has_offence"] == 1].copy()
df_type["top_offence"] = df_type["top_offence"].astype(str).str.strip()

# remove none if any
df_type = df_type[df_type["top_offence"].str.lower() != "none"].copy()
if df_type.empty:
    raise ValueError("Type dataset is empty after filtering (has_offence==1 and top_offence != 'none').")

# merge rare classes
counts = df_type["top_offence"].value_counts()
rare = counts[counts < MIN_SAMPLES_PER_CLASS].index
df_type["top_offence"] = df_type["top_offence"].replace(rare, "other_offence")

train_t = df_type[df_type["year"] < max_year].copy()
test_t  = df_type[df_type["year"] == max_year].copy()

print("\nType train:", train_t.shape, "Type test:", test_t.shape)
print("Type class counts (after merge):")
print(df_type["top_offence"].value_counts())

X_train_t = train_t[FEATURE_COLUMNS]
y_train_t = train_t["top_offence"]

X_test_t = test_t[FEATURE_COLUMNS]
y_test_t = test_t["top_offence"]

type_model = Pipeline([
    ("prep", preprocessor),
    ("clf", RandomForestClassifier(
        n_estimators=800,
        max_depth=20,
        min_samples_leaf=1,
        class_weight="balanced",
        random_state=RANDOM_STATE,
        n_jobs=-1,
    )),
])

print("\nTraining type_model_v4...")
type_model.fit(X_train_t, y_train_t)

type_pred = type_model.predict(X_test_t)
print("\nTYPE MODEL REPORT")
print(classification_report(y_test_t, type_pred, zero_division=0))

joblib.dump(type_model, TYPE_MODEL_PATH)
print("Saved:", TYPE_MODEL_PATH)

# ===============================
# SAVE META JSON
# ===============================
meta = {
    "version": "v4",
    "history_len": 6,
    "risk_threshold": RISK_THRESHOLD,
    "categorical_features": CATEGORICAL_FEATURES,
    "numeric_features": NUMERIC_FEATURES,
    "feature_columns": FEATURE_COLUMNS,
    "type_classes": sorted(df_type["top_offence"].unique().tolist()),
    "train_test_split": {
        "method": "time_based_last_year_test",
        "train_years": f"< {max_year}",
        "test_year": int(max_year),
    },
    "settings": {
        "random_state": RANDOM_STATE,
        "min_samples_per_class": MIN_SAMPLES_PER_CLASS,
    },
    "notes": [
        "Type model trained only on offence months (has_offence=1).",
        "Rows with top_offence='none' removed.",
        "Rare classes merged into 'other_offence'.",
        "Risk model probability calibrated (sigmoid)."
    ]
}

with open(META_PATH, "w", encoding="utf-8") as f:
    json.dump(meta, f, indent=2)

print("Saved:", META_PATH)
print("\n✅ Finished training v4 models.")


ValueError: Missing required columns: ['n_cases']